# Assignments: Househunters

**Introduction**

HouseHunters is a website advertising houses. Currently, several real estate agencies offer the houses that are sold through their office. In the future, HouseHunters wants to be a platform for private sellers as well. As the asking price generally is different from the value of a house, HouseHunters wants to give private sellers an indication of the asking price of their house according to asked prices of sold houses in the past. Sellers can take this into account when deciding on their selling price.

**Assignments**

HouseHunters wants to accurately predict housing prices and be able to understand which characteristics drive the price of a house. In order to do this you will use different modeling techniques. After each theoretical part about a modeling technique, you will apply this technique on the househunters data. The first of the script is about loading data, exploring the data etc. Go through this part quickly to understand what is happening. You might have to change the directory from which the data is looaded, or change the file filename. The assignments start from the header 'Assignment 1: Linear Regression'. 

## Install packages and load data

In [ ]:
# a) Install packages

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import pandas as pd       # 'as' := we abbreviate the package for common use
pd.options.mode.chained_assignment = None
import numpy as np
import seaborn as sns
import random
import os
import math
import matplotlib.pyplot as plt 
import datetime
import itertools

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import statsmodels.api as sm
import xgboost as xgb

from sklearn.linear_model import LogisticRegression #Logistic regression
from sklearn.metrics import confusion_matrix 
from sklearn.metrics import accuracy_score 
from sklearn import metrics #ROC curve

In [ ]:
# Define function for the MAPE

def mean_absolute_percentage_error(y_true, y_pred):
    return np.mean(np.abs((y_true-y_pred)/y_true))*100

In [ ]:
# b) Load data

# change directory if necessary
inputdata = pd.read_excel("190322 - HouseTable_vDef_excel.xlsx")

#    Get an overview of the data
inputdata.head(10)

## Data exploration

#### Variable exploration

In [ ]:
# a) Get an overview of the data
#    Get the number of rows and columns
print('(nrow, ncol):', inputdata.shape)     

#    Show a brief summary of the numeric variables
inputdata.describe()                        # min/max, count, mean, std and percentiles

In [ ]:
# b) Check data type of each variable
print(pd.DataFrame(inputdata.dtypes, columns=['Datatype']))

In [ ]:
# c) Get an overview of the NULLS in the dataset
nulls = pd.DataFrame(inputdata.isnull().sum(), columns=['# NULLS'])        # Number of NULLS 

lst={}                                                                     # Number of NULLS as a percentage
for col in inputdata.columns:                                       
    lst[col]=np.sum(inputdata.loc[:,col].isnull())/len(inputdata.loc[:,col])
percNulls = pd.DataFrame(pd.Series(lst), columns=['% NULLS'])

print(pd.concat([nulls, percNulls], axis=1))

In [ ]:
# d) Plot distribution of target variable
target = 'Price'

plt.figure(figsize=(15,8))
plt.hist((inputdata[target]), bins=250, color = 'blue', edgecolor = 'blue')
plt.title('Distribution of target variable')
plt.xlabel(target)
plt.ylabel('#')

#### Explore relations with target variable 'Price'

In [ ]:
# e) Create scatter plot to see the relation between numerical (not dummy) variables and target variable
numericals = ['ConstructionYear'
              ,'CapacityHouse_m3'
              ,'LivingArea_m2'
              ,'#Bedrooms'
              ,'StatusScore'
              ,'StatusRank'
              ,'Urbanity_class'
              ,'Avg_house_value_WOZ_1000euros'
              ,'Num_benefit_total'
              ,'Num_WWB'
              ,'Num_AO'
              ,'Num_WW'
              ,'Num_AOW'
              ,'Municipality_Distance_childDaycare_km'
              ,'Municipality_Distance_largeSupermarket_km'
              ,'Municipality_Distance_hospital_km'
              ,'Municipality_Distance_trainstation_km'
              ,'Avg_WOZ_m2'
             ]  


plt.figure(figsize=(20,60))
for i, column in enumerate(numericals):
    plt.subplot(math.ceil(len(numericals)/2), 2, i+1)
    x = inputdata[column]
    y = inputdata[target]
    plt.scatter(x, y, marker='o', cmap='blue')
    plt.xlabel(column)
    plt.ylabel('Price')

In [ ]:
import math
import seaborn as sns
import matplotlib.pyplot as plt

dummies = [
    'ResidentialNeighborhood',
    'QuietRoad',
    'Garden',
    'FirePlace',
    'Balcony',
    'Attic',
    'Back'
]

plt.figure(figsize=(20,20))
for i, column in enumerate(dummies):
    plt.subplot(math.ceil(len(dummies)/3), 3, i+1)
    sns.barplot(x=inputdata[column], y=inputdata[target], palette='Blues')
    plt.title(column)
plt.tight_layout()
plt.show()


In [ ]:
# g) Create barplot to see the relation between the categorical variables and the target variable

plt.figure(figsize=(15,7))
sns.barplot(x=inputdata['Urbanity_class'], y=inputdata[target], palette='Blues')

plt.figure(figsize=(15,7))
sns.barplot(x=inputdata['HouseType'], y=inputdata[target], palette='Blues')

plt.figure(figsize=(15,7))
sns.barplot(x=inputdata['Province'], y=inputdata[target], palette='Blues')

# Additional variables with hue
plt.figure(figsize=(15,7))
sns.barplot(x=inputdata['HouseType'], y=inputdata[target], hue=inputdata['Balcony'], palette='Blues')

plt.figure(figsize=(15,7))
sns.barplot(x=inputdata['Province'], y=inputdata[target], hue=inputdata['Balcony'], palette='Blues')


In [ ]:
# Compute correlations only over numeric columns
corrmat = inputdata.select_dtypes(include='number').corr().round(2)

# Print correlation with the target variable
print(corrmat['Price'].sort_values(ascending=False))


In [ ]:
#    Show correlation matrix to look for mutual correlations
columns = ['Price'
          # ,'ConstructionYear'
           ,'CapacityHouse_m3'
           ,'LivingArea_m2'
          # ,'ResidentialNeighborhood'
           ,'QuietRoad'
           ,'Garden'
           ,'FirePlace'
           ,'Balcony'
          # ,'Attic'
          # ,'Back'
           ,'#Bedrooms'
           ,'StatusRank'
           ,'StatusScore'
           ,'Urbanity_class'
           ,'Avg_house_value_WOZ_1000euros'
          # ,'Num_benefit_total'
           ,'Num_WWB'
           ,'Num_AO'
           ,'Num_WW'
          # ,'Num_AOW'
          # ,'Municipality_Distance_hospital_km'
           ,'Municipality_Distance_childDaycare_km'
          # ,'Municipality_Distance_largeSupermarket_km'
          # ,'Municipality_Distance_trainstation_km'
           ,'Avg_WOZ_m2'
          ]

correlation = inputdata[columns]
corrmat = correlation.corr().round(2)
f, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(corrmat, square=True, annot=True)

## Data preparation

In [ ]:
print(inputdata.shape)

In [ ]:
# a) Drop rows with a price <= 2000, as these are probably for rent, and price >= 1.500.000, as these are outliers

inputdata = inputdata[~(inputdata['Price']<=3000)]
inputdata = inputdata[~(inputdata['Price']>=1500000)]

In [ ]:
# b) Feature engineering
#    i. Add column Price per meter2
inputdata['Price_m2'] = inputdata['Price']/inputdata['LivingArea_m2']

#    ii. Categorize the houses according to the construction year
inputdata['Age_cat'] = inputdata['ConstructionYear']
inputdata['Age_cat'][inputdata['ConstructionYear']<1940] = 'Before war'
inputdata['Age_cat'][(inputdata['ConstructionYear']>=1940) & (inputdata['ConstructionYear']<1990)] = 'Existing'
inputdata['Age_cat'][inputdata['ConstructionYear']>=1990] = 'New construction'

In [ ]:
#    iii. Create dummy-variable whether a house in the city- or countryside
cityside = ['Noord Holland'
            ,'Zuid Holland'
            ,'Utrecht'
           ]

countryside = ['Zeeland'
              ,'Fryslân'
              ,'Groningen'
              ,'Drenthe'
              ]

inputdata['CitySide'] = inputdata['Province'].isin(cityside).astype(np.int8)
inputdata['CountrySide'] = inputdata['Province'].isin(countryside).astype(np.int8)

# Assignment 1: Linear Regression

Note on the difference between 'statsmodels and scikit'. Linear regression is in its basic form the same in statsmodels and in scikit-learn. 

Scikit-learn uses a machine learning approach where the focus is on achieving the best prediction. Statsmodels has a more statistical approach where the focus is on understanding important variables, statistifical significance etc.

As a consequence, the emphasis in the supporting features of statsmodels is in analysing the training data which includes hypothesis tests and goodness-of-fit measures, while the emphasis in the supporting infrastructure in scikit-learn is on model selection for out-of-sample prediction and therefore cross-validation on "test data".
This points out the distinction, there is still quite a lot of overlap also in the usage. statsmodels also does prediction, and additionally forecasting in a time series context. But, when we want to do cross-validation for prediction in statsmodels it is currently still often easier to reuse the cross-validation setup of scikit-learn together with the estimation models of statsmodels.

In order to understand the significance of the different features you will use statsmodels for this assignment.

In [ ]:
inputdata.head()

In [ ]:
# Check missing values

nulls = pd.DataFrame(inputdata.isnull().sum(), columns=['# NULLS'])        # Number of NULLS 

lst={}                                                                     # Number of NULLS as a percentage
for col in inputdata.columns:                                       
    lst[col]=np.sum(inputdata.loc[:,col].isnull())/len(inputdata.loc[:,col])
percNulls = pd.DataFrame(pd.Series(lst), columns=['% NULLS'])

print(pd.concat([nulls, percNulls], axis=1))

In [ ]:
# possible solution is to drop the NAs
print(inputdata.shape)
inputdata = inputdata.dropna()
inputdata_with_na = inputdata[inputdata['Garden'].isnull()]
print(inputdata.shape)
print(inputdata_with_na.shape)

In [ ]:
# Dummmify the categorical variables
# Save the categorical variable columns

dataProv = inputdata['Province']
dataHouse = inputdata['HouseType']
dataAge = inputdata['Age_cat']
dataUrban = inputdata['Urbanity_class']
dataGarden = inputdata['Garden']

#    Get/add dummies
categoricals = ['Province'
                ,'HouseType'
                ,'Age_cat'
                ,'Urbanity_class'
                ,'Garden'
               ]
inputdata = pd.get_dummies(inputdata, columns=categoricals)

#    Add the original categorical column
inputdata['Province'] = dataProv
inputdata['HouseType'] = dataHouse
inputdata['Age_cat'] = dataAge
inputdata['Urbanity_class'] = dataUrban
inputdata['Garden'] = dataGarden

In [ ]:
inputdata.head()

In [ ]:
# split the data into a train and test set

trainset, testset = train_test_split(inputdata, test_size=0.3, random_state=13)

In [ ]:
trainset.columns.values

In [ ]:
# Design X and y
# ADD INDEPENDENT VARIABLES TO X YOU THINK HAVE (HIGH) IMPACT ON PRICE

X_variables = [#'Price_m2_HouseType'
               #'LivingArea_m2'
               # ,'FirePlace'
               # 'Province_Groningen'
               'Province_Noord Holland'
               ,'StatusScore'
               ,'QuietRoad'
               # ,'Age_cat_Existing'
               ,'Garden'
               ,'HouseType_Detached'
               # ,'Urbanity_class_1'
               #,'Urbanity_class_5'
                 ,'#Bedrooms' 
              ]

X_train = trainset.loc[:, X_variables]
X_test = testset.loc[:, X_variables]

y_train = trainset[target]
y_test = testset[target]

In [ ]:
X_train.head()

Now let's actually train the model

In [ ]:
import statsmodels.formula.api as smf
import statsmodels.api as sm

formula = "Price ~ CapacityHouse_m3 + LivingArea_m2 + C(HouseType) + C(Province) + C(Urbanity_class)"
results = smf.ols(formula=formula, data=trainset).fit()


In [ ]:
# b) Get a complete summary of the model

print(results.summary())

In [ ]:
# c) Make predictions for the test set

# testset must include all columns referenced in the formula
y_test_pred = results.predict(testset)

plt.figure(figsize=(7.5,7.5))
plt.scatter(y_test, y_test_pred, alpha=0.4, marker='o')
plt.plot((0, max(y_test)), (0, max(y_test)), 'r-', label='Real=Predicted')
plt.xlabel('Real values')
plt.ylabel('Predictions')
plt.legend(loc='upper left')
plt.show()


In [ ]:
# d) Model evaluation for the test set

print("R-squared: %.3f" % r2_score(y_test, y_test_pred))
print("Mean absolute error: %.2f" % mean_absolute_error(y_test, y_test_pred))
print("Mean absolute percentage error: %.2f" % mean_absolute_percentage_error(y_test, y_test_pred))
print("Mean squared error: %.2f" % mean_squared_error(y_test, y_test_pred))

# Assignment 2: Logistic Regression

As you might have noticed we encoutered some missing variables for the feature 'Garden'. This is important feature in our model and we would like to use it to its full potential. In the next assignment you test whether a logistic regression could be used to predict a missing categorical feature like 'Garden'.

In [ ]:
# --- Dummify categorical variables ---
X_train_model = pd.get_dummies(X_train_model, drop_first=True)
X_test_model = pd.get_dummies(X_test_model, drop_first=True)

# --- Ensure same columns in test as train ---
X_test_model = X_test_model.reindex(columns=X_train_model.columns, fill_value=0)

# --- Convert all columns to numeric (float) ---
X_train_model = X_train_model.astype(float)
X_test_model = X_test_model.astype(float)

# --- Convert target to numeric ---
y_train = pd.to_numeric(y_train)
y_test = pd.to_numeric(y_test)

# --- Add constant for intercept ---
X_train_model = sm.add_constant(X_train_model)
X_test_model = sm.add_constant(X_test_model)

# --- Fit logistic regression ---
logit_model = sm.Logit(y_train, X_train_model)
result = logit_model.fit()
print(result.summary2())


In [ ]:
preds_log = result.predict(X_test_model)

In [ ]:
#Cast above 0.5 to be binary 1
preds_log = (preds_log > 0.5).astype(int)

In [ ]:
# scikit-learn confusion matrix

confusion_matrix(y_test, preds_log)

In [ ]:
#    Define a formula for the confusion matrix
def plot_confusion_matrix(cm, classes,
                          normalize=False,
                          title='Confusion matrix',
                          cmap=plt.cm.Blues):
    
    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)

    fmt = 'd'
    thresh = cm.max() / 2.
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, format(cm[i, j], fmt),
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")

    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.tight_layout()
    
np.set_printoptions(precision=2)


#   Plot confusion matrix
results = confusion_matrix(y_test, preds_log, sample_weight=None)
plt.figure()
plot_confusion_matrix(results, classes=[0,1],
                      title='Confusion matrix')

accuracy_log = accuracy_score(y_test, preds_log)
print('Accuracy Score :')
print(accuracy_score(y_test, preds_log)) 

In [ ]:
#  Create ROC curve

preds_proba_log = result.predict(X_test_model)

fpr_log, tpr_log, t_log = metrics.roc_curve(y_test, preds_proba_log)
#calculate AUC
roc_auc_log = metrics.auc(fpr_log, tpr_log)


#Plot results
plt.plot([0,1],[0,1],linestyle = '--',lw = 2,color = 'black')
plt.plot(fpr_log, tpr_log, color='blue',
         label=r'ROC log (AUC = %0.2f )' % (roc_auc_log),lw=2, alpha=1)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC')
plt.legend(loc="lower right")
plt.show()

# Assignment 3: XGBoost

In [ ]:
trainset.columns.values

In [ ]:
# Now, correlated features do not have to be implemented yourself as a tree handles with them itself
# Define X and y
# ADD INDEPENDENT VARIABLES TO X YOU THINK HAVE (HIGH) IMPACT ON PRICE
X_variables =  [#'Price_m2_HouseType'
                #'LivingArea_m2'
                # ,'FirePlace'
                # 'Province_Groningen'
                 'Province_Noord Holland'
                 ,'StatusScore'
                 ,'QuietRoad'
                #,'Age_cat_Existing'
                 ,'HouseType_Detached'
                #,'Urbanity_class_1'
                #,'Urbanity_class_5'
                  ,'#Bedrooms' 
              ]

X_train = trainset.loc[:, X_variables]
X_test = testset.loc[:, X_variables]

y_train = trainset[target]
y_test = testset[target]

In [ ]:
# a) Train the model using the training sets
bst = xgb.XGBRegressor(max_depth=3
                      ,min_child_weight=1
                      ,gamma=0
                      ,subsample=1
                      ,colsample_bytree=1
                      ,learning_rate=0.3
                      ,silent=1
                      ,n_estimators=100)
bst.fit(X_train, y_train)

# b) Plot feature importances
fig, ax = plt.subplots(figsize=(15,10))
xgb.plot_importance(bst, importance_type='gain', show_values=False, ax=ax)

# c) Make predictions for the test set
y_test_pred = bst.predict(X_test)
plt.figure(figsize=(7.5,7.5))
plt.scatter(y_test, y_test_pred, alpha=0.4, marker='o')
plt.plot((0, max(y_test)), (0, max(y_test)), 'r-', label='Real=Predicted')
plt.xlabel('Real values')
plt.ylabel('Predictions')
plt.legend(loc='upper left')

In [ ]:
# d) Model evaluation for the test set
print("R-squared: %.3f" % r2_score(y_test, y_test_pred))
print("Mean absolute error: %.2f" % mean_absolute_error(y_test, y_test_pred))
print("Mean absolute percentage error: %.2f" % mean_absolute_percentage_error(y_test, y_test_pred))
print("Mean squared error: %.2f" % mean_squared_error(y_test, y_test_pred))

# Assignment 4: Battle of the models

Time to battle it out! Does linear regression or XGBoost perform better? Do you have an idea why? What features are important in the model?